# 08 · One codebase, three backends (Keras 3)

`omnibias.keras` is written entirely against `keras.ops`, so the **same** layers
run on **TensorFlow, JAX, or PyTorch** — you pick the backend with the
`KERAS_BACKEND` environment variable *before* importing Keras. The polynomial
coefficients come from `omnibias.core`, so every backend is bit-identical.

This notebook:

1. confirms the Keras closed-form `σ⁽ⁿ⁾` matches the PyTorch backend, and
2. trains a small Keras model with a `cmbDense` operator layer via `model.fit`.

Re-run with `KERAS_BACKEND=tensorflow` or `=torch` and the results are the same.

In [ ]:
import os, sys
os.environ.setdefault("KERAS_BACKEND", "jax")   # try "tensorflow" or "torch"
os.environ.setdefault("JAX_ENABLE_X64", "true")
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
from _style import set_style, ACCENT, GOOD, PRIMARY, INK
set_style()

import keras
keras.config.set_floatx("float64")
from omnibias.keras import get_activation, cmbDense

print("Keras", keras.__version__, "· backend:", keras.backend.backend())

## 1. Same closed form as the PyTorch backend

We compute `tanh⁽ⁿ⁾` through the Keras kernel (on whatever backend is active)
and compare to `omnibias.torch`. The difference is float64 round-off.

In [ ]:
import torch
from omnibias.torch.activations.registry import get_activation as torch_get

ks, ts = get_activation("tanh"), torch_get("tanh")
z = np.linspace(-3, 3, 9)
zk = keras.ops.convert_to_tensor(z)
zt = torch.tensor(z, dtype=torch.float64)
for n in range(0, 6):
    kv = keras.ops.convert_to_numpy(ks.fastpath(zk, n))
    tv = ts.fastpath(zt, n).numpy()
    print(f"tanh^({n})  keras[{keras.backend.backend()}] vs torch  "
          f"max|Δ| = {np.max(np.abs(kv - tv)):.1e}")

## 2. Train a Keras model with a `cmbDense` operator layer

`cmbDense` is a drop-in `Dense` whose features pass through a typed
`OperatorBlock`. It trains with the standard Keras `fit` loop on any backend.

In [ ]:
keras.utils.set_random_seed(0)
x = np.linspace(-3, 3, 256).reshape(-1, 1)
y = np.sin(2 * x)

model = keras.Sequential([
    keras.layers.Input(shape=(1,)),
    cmbDense(32, op="grad", base="tanh"),
    keras.layers.Dense(32, activation="tanh"),
    keras.layers.Dense(1),
])
model.compile(optimizer=keras.optimizers.Adam(1e-2), loss="mse")
hist = model.fit(x, y, epochs=200, batch_size=64, verbose=0)
pred = model.predict(x, verbose=0)

fig, (axl, axr) = plt.subplots(1, 2, figsize=(11, 4.2))
axl.semilogy(hist.history["loss"], color=PRIMARY); axl.set_xlabel("epoch")
axl.set_ylabel("MSE"); axl.set_title(f"Training ({keras.backend.backend()} backend)")
axr.plot(x, y, color=ACCENT, lw=3, alpha=0.6, label="target sin(2x)")
axr.plot(x, pred, color=INK, ls="--", label="cmbDense model")
axr.set_xlabel("x"); axr.set_title("Fit"); axr.legend()
plt.tight_layout(); plt.show()
print("final train MSE =", hist.history["loss"][-1])

## Takeaway

The same `omnibias.keras` layers — with bit-identical closed-form derivatives —
run and train on TensorFlow, JAX, or PyTorch. Set `KERAS_BACKEND` and nothing
else changes.

That wraps the gallery. Back to **[notebook 01](01_closed_form_derivatives.ipynb)**
or the [README](../README.md).